# **ACID - PART 6 - Extract Features**


In [ ]:
# Import required modules
import numpy as np
import pandas as pd
import os
from pathlib import Path
import skimage
import dask.array as da
import dask
from acid.utils.listdirNHF import listdirNHF
from acid.image_processing.rescale_intensity import quantize_image
from acid.feature_extraction.measure_haralick import (
    glcm_feature_map,
    haralick_compute_feature_count,
)
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops


- boundary should be set to none for glcm feature map

- note the rescaling of the image intensities


In [ ]:
from dask.distributed import Client

client = Client()
print(client)

In [ ]:
client

In [ ]:
input_dir_image = "data/proc/fov_proc"

# input_dir_mask = (r"Z:\AlessandroUlivi_Data\projects\ACID\develop\260326_parallilization_strategy\seg")
input_dir_mask = "data/proc/segmentation"

filenames = listdirNHF(Path(input_dir_image))

In [ ]:
test_file_name = filenames[0]
print(f"Test file name: {test_file_name}")
test_file = skimage.io.imread(os.path.join(Path(input_dir_image), test_file_name))
print(f"Test file shape: {test_file.shape}")
print(f"Test file data type: {test_file.dtype}")
print(f"Max. value in test file: {np.amax(test_file)}")
print(f"Min. value in test file: {np.amin(test_file)}")
print(f"Num. dimensions in test file: {test_file.ndim}")

In [ ]:
@dask.delayed
def load(filename, input_dir_image=input_dir_image, input_dir_mask=input_dir_mask):
    """
    Load one 5-channel image and its matching label mask.
    The image and mask filenames are expected to match.
    """
    image = skimage.io.imread(os.path.join(Path(input_dir_image), filename))
    mask = skimage.io.imread(os.path.join(Path(input_dir_mask), filename))
    return filename, image, mask


def quantize_object_crop(crop, object_mask, levels=8, pmin=1, pmax=99):
    """
    Quantize only the pixels inside one segmented object.
    Pixels outside the object are left as 0 so the crop remains 2D for GLCM.
    """
    values = crop[object_mask]
    quantized = np.zeros(crop.shape, dtype=np.uint8)

    if values.size == 0:
        return quantized

    lo, hi = np.percentile(values, (pmin, pmax))
    if hi <= lo:
        quantized[object_mask] = 0
        return quantized

    clipped = np.clip(crop, lo, hi)
    scaled = (clipped - lo) / (hi - lo)
    scaled = np.floor(scaled * (levels - 1)).astype(np.uint8)
    quantized[object_mask] = scaled[object_mask]
    return quantized


def measure_object_haralick(
    filename,
    image,
    mask,
    channel_axis=0,
    levels=8,
    props=None,
    distances=None,
    angles=None,
    pmin=1,
    pmax=99,
    graycomatrix_kwargs=None,
):
    """
    Fast object-level Haralick measurement.

    Instead of computing a sliding-window feature map for every pixel, this computes
    one GLCM per segmented object and channel. This is much cheaper and gives one
    feature row per cell.
    """
    if props is None:
        props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
    if distances is None:
        distances = [1]
    if angles is None:
        angles = [0]
    if graycomatrix_kwargs is None:
        graycomatrix_kwargs = {"symmetric": True, "normed": True}

    image_ch_last = np.moveaxis(image, channel_axis, -1) if isinstance(channel_axis, int) else image[..., None]
    label_image = np.squeeze(mask).astype(np.int32)

    rows = []
    for region in regionprops(label_image, intensity_image=image_ch_last):
        minr, minc, maxr, maxc = region.bbox
        object_mask = region.image
        object_row = {
            "file_name": filename,
            "label": region.label,
            "area": region.area,
            "centroid_y": region.centroid[0],
            "centroid_x": region.centroid[1],
            "bbox_min_row": minr,
            "bbox_min_col": minc,
            "bbox_max_row": maxr,
            "bbox_max_col": maxc,
        }

        for ch in range(image_ch_last.shape[-1]):
            crop = region.image_intensity[..., ch]
            object_values = crop[object_mask]
            object_row[f"ch{ch}_intensity_mean"] = float(np.mean(object_values)) if object_values.size else np.nan
            object_row[f"ch{ch}_intensity_std"] = float(np.std(object_values)) if object_values.size else np.nan
            object_row[f"ch{ch}_intensity_min"] = float(np.min(object_values)) if object_values.size else np.nan
            object_row[f"ch{ch}_intensity_max"] = float(np.max(object_values)) if object_values.size else np.nan

            quantized = quantize_object_crop(crop, object_mask, levels=levels, pmin=pmin, pmax=pmax)
            glcm = graycomatrix(
                quantized,
                distances=distances,
                angles=angles,
                levels=levels,
                **graycomatrix_kwargs,
            )

            for prop in props:
                values = graycoprops(glcm, prop)
                for d_idx, _ in enumerate(distances):
                    for a_idx, _ in enumerate(angles):
                        object_row[f"ch{ch}_haralick_{prop}_d{d_idx}_a{a_idx}"] = float(values[d_idx, a_idx])

        rows.append(object_row)

    return pd.DataFrame(rows)


@dask.delayed
def process_file_object_features(
    loaded,
    channel_axis=0,
    levels=8,
    props=None,
    distances=None,
    angles=None,
    pmin=1,
    pmax=99,
    graycomatrix_kwargs=None,
):
    filename, image, mask = loaded
    print(f"measuring object features for {filename}")
    return measure_object_haralick(
        filename=filename,
        image=image,
        mask=mask,
        channel_axis=channel_axis,
        levels=levels,
        props=props,
        distances=distances,
        angles=angles,
        pmin=pmin,
        pmax=pmax,
        graycomatrix_kwargs=graycomatrix_kwargs,
    )


def f_object_haralick(
    filenames,
    channel_axis=0,
    levels=8,
    props=None,
    distances=None,
    angles=None,
    pmin=1,
    pmax=99,
    graycomatrix_kwargs=None,
):
    tasks = []
    for filename in filenames:
        loaded = load(filename)
        task = process_file_object_features(
            loaded,
            channel_axis=channel_axis,
            levels=levels,
            props=props,
            distances=distances,
            angles=angles,
            pmin=pmin,
            pmax=pmax,
            graycomatrix_kwargs=graycomatrix_kwargs,
        )
        tasks.append(task)
    return tasks


In [ ]:
channel_axis = 0
levels = 8
haralick_props = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM",
]
distances = [1]
angles = [0]
pmin = 1
pmax = 99
graycomatrix_kwargs = {"symmetric": True, "normed": True}

# Use a short slice while testing, then switch back to filenames for the full run.
filenames_to_process = filenames


In [ ]:
feature_tables = dask.compute(
    *f_object_haralick(
        filenames_to_process,
        channel_axis=channel_axis,
        levels=levels,
        props=haralick_props,
        distances=distances,
        angles=angles,
        pmin=pmin,
        pmax=pmax,
        graycomatrix_kwargs=graycomatrix_kwargs,
    )
)

features_df = pd.concat(feature_tables, ignore_index=True)
features_df


In [ ]:
output_feature_directory = "data/proc/features"
os.makedirs(output_feature_directory, exist_ok=True)

feature_file_name = "ACID_object_haralick_features_part_6.csv"
features_df.to_csv(os.path.join(output_feature_directory, feature_file_name), index=False)
print(f"saved {features_df.shape[0]} object rows to {os.path.join(output_feature_directory, feature_file_name)}")


the file is a 2720x2720 field of view with 7 channels.

The first of the 7 channels is the segmentation mask.

Objects are individual cells.

The remaining 6 channels are different imaged structures / imaging modalities.

In [ ]:
# real_data_haralick_features = measure_haralick_features(image=input_real_image[...,1:3],
#                                              label_image=input_real_image[...,0],
#                                              props=['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'],
#                                              distances=[5, 15, 49],
#                                              angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
#                                              channel_axis=-1,
#                                              window_shape=50,
#                                              glcm_daskbag_kwargs={'npartitions': 10})

# real_data_haralick_features

# # single channel,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 10 partitions -> 8m 16s
# # double channels,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 12 partitions -> 15m 33s

